# V16 · Rechner und Betriebssystem: Dateien der Fertigungszelle durchsuchen · Vertiefung zum Training

**Wo passt das Notebook hin?** Erst die Lerneinheit und das Programmierlabor, dann das Training zu Hause (Browser, ca. 60 Min). Dieses Notebook ist die **Vertiefung** und läuft im **Codespace** oder in Jupyter auf deinem Rechner. Du übst mit den Dateien der Fertigungszelle FZ-3 aus dem Labor: `pathlib` (Pfade bauen und zerlegen, Ordner auflisten, Dateien mit `glob` und `rglob` suchen), `sorted`, Dateigrößen mit `stat()`, die ältere Schreibweise mit `os` (nur lesen) und die Laufzeit mit `time.perf_counter()`. Es nutzt Python aus V01 bis V15: Funktionen, Listen, Dictionaries, Dateien, `csv`, List Comprehension.

**Die Dateien der Zelle bringt das Notebook selbst mit.** Die Werkzeug-Zelle unten legt den Ordner `zelle/` im Ordner des Notebooks an. Sie entpackt dafür `zelle.zip`, wenn sie neben dem Notebook liegt (oder unter `../assets/data/v16/zelle.zip` bzw. `site/assets/data/v16/zelle.zip`, so liegt sie im Repository). Findet sie keine Datei, erzeugt sie den Ordner mit demselben Inhalt selbst. Du musst nichts herunterladen. Deine eigenen Ordner durchsuchst du in Aufgabe 3.5.

**Der Arbeitsablauf bei jeder Aufgabe:** (1) Schreibe bei Vorhersage-Aufgaben deine Vorhersage als Kommentar auf, (2) bearbeite die `TODO`-Stellen, (3) führe die Aufgabenzelle aus, (4) führe die Prüfzelle darunter aus. Die Prüfzellen sind keine Musterlösungen, sondern zeigen nur, ob dein Ergebnis stimmt. Die Lösungen stehen eingeklappt am Ende, erst nach einem ehrlichen Versuch öffnen.

**Achtung, Betriebssysteme unterscheiden sich:** Unter Windows trennt `\` die Ordner, unter Linux und macOS `/`, und Windows sortiert Pfade ohne Rücksicht auf Groß- und Kleinschreibung. `Path` baut den Pfad richtig und `sorted(...)` legt die Reihenfolge fest. Verlass dich nie auf die Reihenfolge von `glob`, `rglob` oder `iterdir`. Die Aufgaben sind so gestellt, dass sie auf allen Systemen dieselben Ergebnisse haben.

**Regeln:** Nur Python aus V01 bis V16. Löschen oder Verändern von Dateien gehört nicht dazu (siehe Code-Review 3.4). KI nur als Erklärer. **Datenschutz:** Keine Firmen- oder Kundendaten in Prompts; hier genügen die Übungsdaten der Zelle.

In [ ]:
# Werkzeug-Zelle: einmal ausführen, nicht verändern.
# Sie legt den Ordner zelle/ an (die Dateien der Fertigungszelle FZ-3) und definiert die Selbsttests.
import os
import random
import zipfile
from pathlib import Path

def _zelle_dateien():
    tage = ["2026-11-16", "2026-11-17", "2026-11-18", "2026-11-19", "2026-11-20"]
    meldungen = {
        "D2": [("INFO", "Spindel an, 1450 1/min"), ("INFO", "Programm WELLE_20H6 gestartet"), ("INFO", "Teil fertig"),
               ("WARN", "Werkzeugverschleiss T4 ueber 80 %"), ("ERROR", "Kuehlmitteldruck zu niedrig")],
        "F1": [("INFO", "Programm FLANSCH_80 gestartet"), ("INFO", "Teil fertig"), ("WARN", "Spaenebehaelter fast voll"),
               ("ERROR", "Achse Z Schleppfehler")],
        "R1": [("INFO", "Greifer zu"), ("INFO", "Greifer auf"), ("INFO", "Teil abgelegt"), ("WARN", "Lichtgitter unterbrochen"),
               ("ERROR", "Not-Halt ausgeloest")],
        "M1": [("INFO", "Messung gestartet"), ("INFO", "Messung i. O."), ("WARN", "Messung n. i. O."),
               ("ERROR", "Taster nicht kalibriert")],
    }
    gewicht = {"INFO": 30, "WARN": 4, "ERROR": 1}
    rng = random.Random(16)
    out = {}
    for maschine in meldungen:
        for tag in tage:
            zeilen = []
            for i in range(rng.randint(40, 70)):
                stufe_meld = rng.choices(meldungen[maschine], [gewicht[s] for s, _ in meldungen[maschine]])[0]
                h, m, s = 6 + i * 14 // 60 % 16, (i * 14) % 60, rng.randint(0, 59)
                zeilen.append(f"{tag} {h:02d}:{m:02d}:{s:02d};{stufe_meld[0]};{stufe_meld[1]}")
            out[f"zelle/maschinen/{maschine}/logs/{tag}.log"] = "\n".join(zeilen) + "\n"
    out["zelle/nc-programme/WELLE_20H6.nc"] = "%\nO1001 (WELLE 20 H6)\nG21 G90 G95\nT0404 M06\nG96 S180 M03\nG00 X22 Z2\nG01 X20 Z-60 F0.15\nG00 X100 Z100\nM30\n%\n"
    out["zelle/nc-programme/FLANSCH_80.nc"] = "%\nO2001 (FLANSCH 80)\nG21 G90 G94\nT1 M06\nS3000 M03\nG81 X30 Y0 Z-12 R2 F200\nX15 Y25.98\nX-15 Y25.98\nG80 M30\n%\n"
    out["zelle/nc-programme/alt/WELLE_20H6_v1.nc"] = "%\nO1000 (ALTE VERSION - NICHT VERWENDEN)\nM30\n%\n"
    messung = ["teil;durchmesser_mm;ergebnis"]
    for teil in range(1, 21):
        d = rng.gauss(20.004, 0.005)
        ergebnis = "iO" if 19.98 <= d <= 20.02 else "niO"
        messung.append(f"{teil};{d:.4f};{ergebnis}")
    out["zelle/messprotokolle/M1_D2_KW47.csv"] = "\n".join(messung) + "\n"
    out["zelle/LIESMICH.txt"] = ("Fertigungszelle FZ-3 (Halle 2)\nD2 Drehmaschine, F1 Fraesmaschine, R1 Roboter, M1 Messmaschine\n"
                                 "Logs: maschinen/<Maschine>/logs/<Datum>.log, Format: Zeit;Stufe;Meldung\n")
    return out

def zelle_bereitstellen():
    if os.path.isdir("zelle"):
        return "Der Ordner zelle/ ist schon da."
    for zip_pfad in ("zelle.zip", "../assets/data/v16/zelle.zip", "site/assets/data/v16/zelle.zip"):
        if os.path.isfile(zip_pfad):
            with zipfile.ZipFile(zip_pfad) as z:
                z.extractall(".")
            return "zelle.zip entpackt (" + zip_pfad + ")."
    for name, text in _zelle_dateien().items():
        ziel = Path(name)
        ziel.parent.mkdir(parents=True, exist_ok=True)
        with open(ziel, "w", encoding="utf-8", newline="") as datei:
            datei.write(text)
    return "Keine zelle.zip gefunden: den Ordner zelle/ selbst erzeugt (gleicher Inhalt)."

print(zelle_bereitstellen())

# Sollwerte der Selbsttests (aus den Dateien oben)
SOLL = {
    "logs": 20, "kib": 47.2197265625,
    "error": {"D2": 5, "F1": 4, "M1": 11, "R1": 7}, "warn": {"D2": 13, "F1": 14, "M1": 20, "R1": 15},
    "erster": {"D2": "2026-11-16 11:22:00 Kuehlmitteldruck zu niedrig", "F1": "2026-11-16 13:42:24 Achse Z Schleppfehler",
               "M1": "2026-11-16 14:24:20 Taster nicht kalibriert", "R1": "2026-11-16 09:02:27 Not-Halt ausgeloest"},
    "mittel": 20.00229, "max": 20.0111,
}

def pruefe(ist, soll, text):
    if ist == soll:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")

def pruefe_nah(ist, soll, text, toleranz=1e-9):
    # Für Kommazahlen: "gleich" heißt hier, die Abweichung ist winzig (Rundungsfehler!).
    if isinstance(ist, (int, float)) and abs(ist - soll) <= toleranz:
        print("OK: " + text)
    else:
        print("FEHLER: " + text + " (erwartet " + str(soll) + ", bei dir steht " + str(ist) + ")")

def pruefe_ja(bedingung, text):
    if bedingung:
        print("OK: " + text)
    else:
        print("FEHLER: " + text)


## Stufe 1 – Aufwärmen

### 1.1 Vorhersagen: Pfade und Pfadteile
Ein `Path` ist ein Pfad als Objekt. Der Schrägstrich `/` verbindet Teile, `.name`, `.stem`, `.suffix` und `.parent` zerlegen ihn, `.exists()` fragt, ob es den Pfad wirklich gibt. Schreibe zuerst auf, was jede Zeile ausgibt (die `parent`-Zeilen sind die kniffligen), und führe dann aus.

<details><summary>Tipp anzeigen</summary>

Tipp: `.parent` ist der Ordner **darüber**, also `logs`. Davon wieder `.parent` ist `D2`. Ein Pfad, den es nicht gibt, erzeugt hier keinen Fehler: `exists()` sagt nur `False`.
</details>

In [ ]:
# Meine Vorhersage: ...
from pathlib import Path

p = Path("zelle") / "maschinen" / "D2" / "logs" / "2026-11-16.log"
print(p.name)
print(p.stem)
print(p.suffix)
print(p.parent.name)
print(p.parent.parent.name)
print(p.exists())
print(Path("zelle/gibt-es-nicht").exists())

### 1.2 Selbsttest: einen Pfad aus Teilen bauen
Baue den Pfad zur Log-Datei des Roboters R1 vom 18. November 2026 (`zelle/maschinen/R1/logs/2026-11-18.log`) aus **Teilen** mit dem Schrägstrich, nicht als einen langen Text. Die Datei muss es wirklich geben.

<details><summary>Tipp anzeigen</summary>

Tipp: `Path("zelle") / "maschinen" / "R1" / ...` und so weiter, jeder Ordner und der Dateiname ist ein Text in Anführungszeichen.
</details>

In [ ]:
from pathlib import Path

log = None          # TODO: Pfad zelle/maschinen/R1/logs/2026-11-18.log mit / bauen

In [ ]:
pruefe_ja(isinstance(log, Path), "log ist ein Path")
pruefe_ja(isinstance(log, Path) and log.name == "2026-11-18.log", "Dateiname stimmt")
pruefe_ja(isinstance(log, Path) and log.parent.parent.name == "R1", "gehört zur Maschine R1")
pruefe_ja(isinstance(log, Path) and log.exists(), "die Datei gibt es wirklich")

### 1.3 Selbsttest: Pfadteile bestimmen
Aus einem Pfad willst du die Teile herausholen: die Maschine (Name des Ordners zwei Ebenen über der Datei), das Datum (Dateiname **ohne** Endung) und die Endung.

<details><summary>Tipp anzeigen</summary>

Tipp: `p.parent.parent.name` liefert die Maschine, `p.stem` das Datum, `p.suffix` die Endung mit Punkt.
</details>

In [ ]:
from pathlib import Path

p = Path("zelle/maschinen/M1/logs/2026-11-19.log")
maschine = None     # TODO: M1
datum = None        # TODO: 2026-11-19
endung = None       # TODO: .log

In [ ]:
pruefe(maschine, "M1", "Maschine")
pruefe(datum, "2026-11-19", "Datum ohne Endung")
pruefe(endung, ".log", "Endung mit Punkt")

### 1.4 Vorhersagen: die ältere Schreibweise mit os
In älterem Code und in KI-Antworten findest du oft das Modul `os`. Du musst es nicht selbst schreiben, aber **lesen** können. Was gibt jede Zeile aus? Vergleiche mit `pathlib`.

<details><summary>Tipp anzeigen</summary>

Tipp: `os.path.join` fügt Teile mit dem Trennzeichen des Betriebssystems zusammen, genau wie `/` bei `Path`. `os.listdir` liefert eine Liste von **Namen** (Text), keine Path-Objekte, und die Reihenfolge ist nicht festgelegt, deshalb `sorted`.
</details>

In [ ]:
# Meine Vorhersage: ...
import os
from pathlib import Path

print(os.path.join("zelle", "maschinen", "D2"))
print(os.path.join("zelle", "maschinen", "D2") == str(Path("zelle") / "maschinen" / "D2"))
print(sorted(os.listdir("zelle")))
print(type(os.getcwd()))

## Stufe 2 – Anwenden

### 2.1 Einen Ordner auflisten
Lege in `eintraege` eine **sortierte Liste der Namen** (Texte) aller Einträge im Ordner `zelle` ab (Dateien und Ordner). Die Reihenfolge von `iterdir()` ist nicht festgelegt, `sorted` macht sie fest.

<details><summary>Tipp anzeigen</summary>

Tipp: List Comprehension wie in V10: `[p.name for p in Path("zelle").iterdir()]`, und das Ganze in `sorted(...)`.
</details>

In [ ]:
from pathlib import Path

eintraege = None    # TODO: sortierte Liste der Namen der Einträge in zelle

In [ ]:
pruefe(eintraege, ["LIESMICH.txt", "maschinen", "messprotokolle", "nc-programme"], "vier Einträge, sortiert")

### 2.2 Dateien finden mit glob
Das Sternchen `*` steht für einen beliebigen Namen, ein Stern je Ordnerebene. Finde (a) **alle** Log-Dateien aller Maschinen und (b) nur die der Fräsmaschine F1. Beides **sortiert**.

<details><summary>Tipp anzeigen</summary>

Tipp: `zelle.glob("maschinen/*/logs/*.log")` findet alle. Für F1 ersetzt du den mittleren Stern durch den Namen `F1`. Vergiss `sorted(...)` nicht.
</details>

In [ ]:
from pathlib import Path

zelle = Path("zelle")
alle_logs = None    # TODO: sortierte Liste aller Log-Dateien
logs_f1 = None      # TODO: sortierte Liste nur der Logs von F1

In [ ]:
pruefe(len(alle_logs) if alle_logs else -1, 20, "20 Log-Dateien (4 Maschinen mal 5 Tage)")
pruefe(len(logs_f1) if logs_f1 else -1, 5, "5 Log-Dateien von F1")
pruefe_ja(bool(logs_f1) and all(p.parent.parent.name == "F1" for p in logs_f1), "alle gehören zu F1")
pruefe_ja(bool(alle_logs) and alle_logs == sorted(alle_logs), "sortiert")

### 2.3 NC-Programme: glob gegen rglob
Im Ordner `nc-programme` liegen zwei Programme, im Unterordner `alt` ein drittes (eine alte Version). Hole (a) die **Namen** (nur die Namen, sortiert) der NC-Programme **direkt** im Ordner mit `glob` und (b) die Namen **aller** NC-Programme, auch in Unterordnern, mit `rglob`.

<details><summary>Tipp anzeigen</summary>

Tipp: `Path("zelle/nc-programme").glob("*.nc")` bleibt im Ordner, `rglob("*.nc")` geht durch alle Unterordner. Aus den Pfaden machst du Namen mit `.name`; sortiere die **Namen**.
</details>

In [ ]:
from pathlib import Path

ordner = Path("zelle") / "nc-programme"
direkt = None       # TODO: sortierte Liste der Namen, nur direkt im Ordner
alle_nc = None      # TODO: sortierte Liste der Namen, auch in Unterordnern

In [ ]:
pruefe(direkt, ["FLANSCH_80.nc", "WELLE_20H6.nc"], "direkt: zwei Programme")
pruefe(alle_nc, ["FLANSCH_80.nc", "WELLE_20H6.nc", "WELLE_20H6_v1.nc"], "rglob: auch die alte Version in alt/")

### 2.4 Dateigrößen: Bytes in KiB
Summiere die Größe **aller** Log-Dateien (in Bytes, `stat().st_size`) und rechne sie in KiB um (1 KiB = 1024 Bytes). Finde außerdem die **größte** Log-Datei (als `Path`).

<details><summary>Tipp anzeigen</summary>

Tipp: Schleife über `sorted(zelle.glob(...))`, Summe mit `gesamt_bytes = gesamt_bytes + log.stat().st_size`. Für die größte Datei merkst du dir die bisher größte Größe und die Datei dazu, wie beim Maximum in V06.
</details>

In [ ]:
from pathlib import Path

zelle = Path("zelle")
gesamt_bytes = 0
groesste = None         # Path der größten Log-Datei
groesse_max = 0
# TODO: Schleife über alle Log-Dateien: Bytes summieren, größte Datei merken
gesamt_kib = gesamt_bytes / 1024
print(f"{gesamt_kib:.1f} KiB, größte Datei: {groesste}")

In [ ]:
pruefe_nah(gesamt_kib, SOLL["kib"], "Gesamtgröße in KiB", 1e-6)
pruefe_ja(isinstance(groesste, Path) and groesste.parent.parent.name == "F1" and groesste.stem == "2026-11-16", "größte Datei: F1, 2026-11-16")

### 2.5 Zeilen mit einer Stufe zählen
Schreibe eine Funktion `zaehle(maschine, stufe)`, die in **allen** Log-Dateien dieser Maschine die Zeilen mit der Stufe (`"INFO"`, `"WARN"` oder `"ERROR"`) zählt und die Anzahl zurückgibt. Eine Zeile sieht so aus: `2026-11-16 11:22:00;ERROR;Kuehlmitteldruck zu niedrig`.

<details><summary>Tipp anzeigen</summary>

Tipp: Die Logs liegen in `zelle/maschinen/<maschine>/logs`. Zeile zerlegen: `zeile.strip().split(";")`, die Stufe steht an Position 1. Zähler vor den Schleifen auf 0, am Ende `return`.
</details>

In [ ]:
from pathlib import Path

def zaehle(maschine, stufe):
    """Zählt die Zeilen mit der Stufe in allen Logs der Maschine."""
    # TODO
    return None

In [ ]:
for m in ("D2", "F1", "M1", "R1"):
    pruefe(zaehle(m, "ERROR"), SOLL["error"][m], "Fehler (ERROR) bei " + m)
pruefe(zaehle("F1", "WARN"), SOLL["warn"]["F1"], "Warnungen (WARN) bei F1")
pruefe(zaehle("X9", "ERROR"), 0, "unbekannte Maschine: 0, kein Absturz")

### 2.6 Messprotokoll auswerten
`zelle/messprotokolle/M1_D2_KW47.csv` hat Semikolon als Trennzeichen (V09): `teil;durchmesser_mm;ergebnis`. Lies die Datei mit dem `csv`-Modul, sammle die Durchmesser als Kommazahlen in `durchmesser` und zähle die Teile mit Ergebnis `niO` (nicht in Ordnung). Berechne dann Mittelwert und Maximum.

<details><summary>Tipp anzeigen</summary>

Tipp: `csv.reader(datei, delimiter=";")`, Kopfzeile mit `next(leser)` überspringen, Spalte 1 ist der Durchmesser (`float(...)`), Spalte 2 das Ergebnis.
</details>

In [ ]:
import csv
from pathlib import Path

pfad = Path("zelle") / "messprotokolle" / "M1_D2_KW47.csv"
durchmesser = []
n_nio = 0
# TODO: Datei lesen, Liste füllen, niO zählen
mittel = None
groesster = None

In [ ]:
pruefe(len(durchmesser), 20, "20 Messwerte gelesen")
pruefe(n_nio, 0, "kein Teil n. i. O.")
pruefe_nah(mittel, SOLL["mittel"], "Mittelwert", 1e-6)
pruefe_nah(groesster, SOLL["max"], "Maximum", 1e-9)

### 2.7 Laufzeit messen: mehrmals und der Bestwert
Miss, wie lange die Suche `sorted(zelle.glob("maschinen/*/logs/*.log"))` dauert: **fünfmal** hintereinander, jede Dauer in **Millisekunden** in die Liste `zeiten`. Der Bestwert (kleinste Zeit) kommt in `bestwert`. Die Werte schwanken von Lauf zu Lauf, weil das Betriebssystem andere Prozesse dazwischenschiebt.

<details><summary>Tipp anzeigen</summary>

Tipp: `start = time.perf_counter()` vor, `ende = time.perf_counter()` nach der Suche. Die Differenz ist in Sekunden, mal 1000 ergibt Millisekunden. Schleife `for versuch in range(5):`.
</details>

In [ ]:
import time
from pathlib import Path

zelle = Path("zelle")
zeiten = []
# TODO: fünfmal messen, jede Dauer in ms an zeiten anhängen
bestwert = None
print(zeiten, bestwert)

In [ ]:
pruefe(len(zeiten), 5, "fünf Messwerte")
pruefe_ja(len(zeiten) == 5 and all(isinstance(z, float) and 0 <= z < 5000 for z in zeiten), "plausible Millisekunden (kleine Kommazahlen)")
pruefe_ja(len(zeiten) == 5 and bestwert == min(zeiten), "Bestwert ist der kleinste Wert")

## Stufe 3 – Vertiefen

### 3.1 Der erste Fehler einer Maschine
Schreibe `erster_fehler(maschine)`: Sie gibt von der **ersten** Zeile mit der Stufe `ERROR` (in der Reihenfolge der sortierten Log-Dateien) den Text `Datum Uhrzeit Meldung` zurück, zum Beispiel `2026-11-16 11:22:00 Kuehlmitteldruck zu niedrig`. Gibt es keinen Fehler (oder keine Maschine), liefert sie `None`.

<details><summary>Tipp anzeigen</summary>

Tipp: Das Datum ist `log.stem`, die Uhrzeit die Zeichen ab Position 11 im Zeitstempel (`teile[0][11:]`), die Meldung `teile[2]`. Beim ersten Treffer sofort `return` (das beendet beide Schleifen). Erst nach den Schleifen `return None`.
</details>

In [ ]:
from pathlib import Path

def erster_fehler(maschine):
    """Gibt den ersten Fehler der Maschine als Text zurück (oder None)."""
    # TODO
    return None

In [ ]:
for m in ("D2", "F1", "M1", "R1"):
    pruefe(erster_fehler(m), SOLL["erster"][m], "erster Fehler bei " + m)
pruefe(erster_fehler("X9"), None, "unbekannte Maschine: None")

### 3.2 Speicher-Ampel (Grenzwerte testen)
Der Arbeitsspeicher (RAM) einer Workstation hat `ram_gb` Gigabyte, die laufenden Programme brauchen zusammen die Summe von `bedarf` (Liste in GB). Die Funktion `ampel(ram_gb, bedarf)` meldet `"ok"`, wenn die Summe höchstens 80 % des RAM beträgt, `"knapp"` bis einschließlich 100 % und sonst `"Auslagerung"` (das Betriebssystem lagert dann auf die SSD aus, das bremst um etwa den Faktor 1000). Teste die **Grenzen** genau: 80 % und 100 % gehören jeweils noch zur besseren Stufe.

<details><summary>Tipp anzeigen</summary>

Tipp: Erst `summe = sum(bedarf)`, dann `if summe <= 0.8 * ram_gb:` ... `elif summe <= ram_gb:` ... `else:`. Das ist der Programmablaufplan aus dem Labor. Eine leere Liste hat die Summe 0.
</details>

In [ ]:
def ampel(ram_gb, bedarf):
    """Meldet "ok", "knapp" oder "Auslagerung"."""
    # TODO
    return None

In [ ]:
pruefe(ampel(16, []), "ok", "keine Programme: ok")
pruefe(ampel(10, [4.0, 4.0]), "ok", "genau 80 %: noch ok")
pruefe(ampel(10, [8.1]), "knapp", "knapp über 80 %: knapp")
pruefe(ampel(10, [6.0, 4.0]), "knapp", "genau 100 %: noch knapp")
pruefe(ampel(10, [10.1]), "Auslagerung", "über 100 %: Auslagerung")
pruefe(ampel(16, [6, 5, 2.4]), "knapp", "13,4 von 16 GB: knapp")

### 3.3 os.walk lesen und mit pathlib umschreiben
Die KI hat die NC-Programme mit `os.walk` gezählt. `os.walk` geht durch den ganzen Baum und liefert je Ordner drei Dinge (Ordnername, Unterordner, Dateien). **Lies** den Code und sage vorher die Ausgabe voraus. Schreibe danach dasselbe mit `pathlib` in `anzahl_pl`.

<details><summary>Tipp anzeigen</summary>

Tipp: Die erste Variable der Schleife ist der aktuelle Ordner, die zweite die Liste der Unterordner, die dritte die Liste der Dateinamen darin. Mit pathlib: `len(sorted(Path("zelle").rglob("*.nc")))`.
</details>

In [ ]:
# Meine Vorhersage: ...
import os
from pathlib import Path

anzahl_os = 0
for ordner, unterordner, dateien in os.walk("zelle"):
    for name in dateien:
        if name.endswith(".nc"):
            anzahl_os = anzahl_os + 1
print(anzahl_os)

anzahl_pl = None    # TODO: dasselbe mit pathlib

In [ ]:
pruefe(anzahl_os, 3, "os.walk zählt drei NC-Programme")
pruefe(anzahl_pl, 3, "pathlib zählt dasselbe")

### 3.4 Code-Review: würdest du das freigeben?
Die KI soll alte Logs der Zelle aufräumen und liefert den Code unten. **Führe ihn nicht aus.** Notiere in einer Markdown-Zelle, was daran gefährlich ist oder fehlt (mindestens drei Punkte). Die Antworten stehen bei den Lösungen.

<details><summary>Tipp anzeigen</summary>

Tipp: Was passiert mit Dateien, die gelöscht sind? Wer entscheidet, welche Logs weg dürfen? Was fehlt an `read_text()`? Und was passiert bei einem Fehler mitten in der Schleife?
</details>

```python
# Vorschlag der KI: "Räumt alte Logs auf"
from pathlib import Path

for log in Path("zelle").rglob("*.log"):
    if "ERROR" not in log.read_text():
        log.unlink()
```

**Meine Review (mindestens drei Punkte):**

- ...
- ...
- ...

### 3.5 Eigene Auswertung: Dateien nach Endung zählen
Zähle in einem Ordner deiner Wahl (zum Beispiel dem Notebook-Ordner oder `zelle`), wie viele Dateien es je Endung gibt, mit einem Dictionary: `{".log": 20, ".nc": 3, ...}`. Gib es aus und schreibe in eine Markdown-Zelle, was dir auffällt.

<details><summary>Tipp anzeigen</summary>

Tipp: Schleife über `ordner.rglob("*")`, `if p.is_file():`, Endung `p.suffix`. Beim ersten Auftreten einer Endung setzt du den Wert auf 0 (V08: `if endung not in zaehler`).
</details>

In [ ]:
from pathlib import Path

ordner = Path("zelle")      # hier deinen Ordner eintragen, z. B. Path(".")
anzahl_je_endung = {}
# Dein Code hier
print(anzahl_je_endung)

**Was mir aufgefallen ist:** ...

## Lösungen
Erst nach einem ehrlichen Versuch öffnen. Es gibt mehrere richtige Wege, entscheidend ist, dass die Prüfzellen grün sind.

<details><summary>Lösung 1.1 Vorhersagen: Pfade und Pfadteile</summary>

Ausgabe: `2026-11-16.log`, `2026-11-16`, `.log`, `logs`, `D2`, `True`, `False`.
</details>

<details><summary>Lösung 1.2 Selbsttest: einen Pfad aus Teilen bauen</summary>

```python
from pathlib import Path

log = Path("zelle") / "maschinen" / "R1" / "logs" / "2026-11-18.log"
```
</details>

<details><summary>Lösung 1.3 Selbsttest: Pfadteile bestimmen</summary>

```python
from pathlib import Path

p = Path("zelle/maschinen/M1/logs/2026-11-19.log")
maschine = p.parent.parent.name
datum = p.stem
endung = p.suffix
```
</details>

<details><summary>Lösung 1.4 Vorhersagen: die ältere Schreibweise mit os</summary>

Ausgabe: `zelle/maschinen/D2` (unter Windows mit `\`), `True`, `['LIESMICH.txt', 'maschinen', 'messprotokolle', 'nc-programme']`, `<class 'str'>`. `os` liefert Texte, `pathlib` liefert Path-Objekte.
</details>

<details><summary>Lösung 2.1 Einen Ordner auflisten</summary>

```python
from pathlib import Path

eintraege = sorted([p.name for p in Path("zelle").iterdir()])
```
</details>

<details><summary>Lösung 2.2 Dateien finden mit glob</summary>

```python
from pathlib import Path

zelle = Path("zelle")
alle_logs = sorted(zelle.glob("maschinen/*/logs/*.log"))
logs_f1 = sorted(zelle.glob("maschinen/F1/logs/*.log"))
```
</details>

<details><summary>Lösung 2.3 NC-Programme: glob gegen rglob</summary>

```python
from pathlib import Path

ordner = Path("zelle") / "nc-programme"
direkt = sorted([p.name for p in ordner.glob("*.nc")])
alle_nc = sorted([p.name for p in ordner.rglob("*.nc")])
```
</details>

<details><summary>Lösung 2.4 Dateigrößen: Bytes in KiB</summary>

```python
from pathlib import Path

zelle = Path("zelle")
gesamt_bytes = 0
groesste = None
groesse_max = 0
for log in sorted(zelle.glob("maschinen/*/logs/*.log")):
    groesse = log.stat().st_size
    gesamt_bytes = gesamt_bytes + groesse
    if groesse > groesse_max:
        groesse_max = groesse
        groesste = log
gesamt_kib = gesamt_bytes / 1024
print(f"{gesamt_kib:.1f} KiB, größte Datei: {groesste}")
```
</details>

<details><summary>Lösung 2.5 Zeilen mit einer Stufe zählen</summary>

```python
from pathlib import Path

def zaehle(maschine, stufe):
    """Zählt die Zeilen mit der Stufe in allen Logs der Maschine."""
    anzahl = 0
    ordner = Path("zelle") / "maschinen" / maschine / "logs"
    for log in sorted(ordner.glob("*.log")):
        with open(log, encoding="utf-8") as datei:
            for zeile in datei:
                if zeile.strip().split(";")[1] == stufe:
                    anzahl = anzahl + 1
    return anzahl
```
</details>

<details><summary>Lösung 2.6 Messprotokoll auswerten</summary>

```python
import csv
from pathlib import Path

pfad = Path("zelle") / "messprotokolle" / "M1_D2_KW47.csv"
durchmesser = []
n_nio = 0
with open(pfad, encoding="utf-8") as datei:
    leser = csv.reader(datei, delimiter=";")
    next(leser)
    for zeile in leser:
        durchmesser.append(float(zeile[1]))
        if zeile[2] == "niO":
            n_nio = n_nio + 1
mittel = sum(durchmesser) / len(durchmesser)
groesster = max(durchmesser)
```
</details>

<details><summary>Lösung 2.7 Laufzeit messen: mehrmals und der Bestwert</summary>

```python
import time
from pathlib import Path

zelle = Path("zelle")
zeiten = []
for versuch in range(5):
    start = time.perf_counter()
    sorted(zelle.glob("maschinen/*/logs/*.log"))
    ende = time.perf_counter()
    zeiten.append(1000 * (ende - start))
bestwert = min(zeiten)
print(zeiten, bestwert)
```
</details>

<details><summary>Lösung 3.1 Der erste Fehler einer Maschine</summary>

```python
from pathlib import Path

def erster_fehler(maschine):
    """Gibt den ersten Fehler der Maschine als Text zurück (oder None)."""
    ordner = Path("zelle") / "maschinen" / maschine / "logs"
    for log in sorted(ordner.glob("*.log")):
        with open(log, encoding="utf-8") as datei:
            for zeile in datei:
                teile = zeile.strip().split(";")
                if teile[1] == "ERROR":
                    return log.stem + " " + teile[0][11:] + " " + teile[2]
    return None
```
</details>

<details><summary>Lösung 3.2 Speicher-Ampel (Grenzwerte testen)</summary>

```python
def ampel(ram_gb, bedarf):
    """Meldet "ok", "knapp" oder "Auslagerung"."""
    summe = sum(bedarf)
    if summe <= 0.8 * ram_gb:
        return "ok"
    elif summe <= ram_gb:
        return "knapp"
    else:
        return "Auslagerung"
```
</details>

<details><summary>Lösung 3.3 os.walk lesen und mit pathlib umschreiben</summary>

Ausgabe: `3` (zwei Programme in `nc-programme`, eines in `alt`). `os.walk` besucht jeden Ordner und gibt Ordnername, Unterordner und Dateinamen darin.

```python
import os
from pathlib import Path

anzahl_os = 0
for ordner, unterordner, dateien in os.walk("zelle"):
    for name in dateien:
        if name.endswith(".nc"):
            anzahl_os = anzahl_os + 1
print(anzahl_os)

anzahl_pl = len(sorted(Path("zelle").rglob("*.nc")))
```
</details>

<details><summary>Lösung 3.4 Code-Review: würdest du das freigeben?</summary>

Gefährlich oder unvollständig:
- `unlink()` löscht **endgültig**, es gibt keinen Papierkorb. Ein Fehler in der Bedingung kostet Betriebsdaten.
- Die Bedingung löscht jede Log-Datei, die **keinen** Fehler enthält, also gerade die gesunden. Gemeint war vermutlich das Gegenteil, oder nur Dateien, die älter als eine Frist sind. Wer entscheidet, welche Logs weg dürfen (Aufbewahrungspflicht, Rückverfolgbarkeit)?
- `read_text()` ohne `encoding="utf-8"` liest je nach Betriebssystem anders. Bei einem Zeichenfehler bricht die Schleife mitten im Löschen ab.
- Es gibt keine Vorschau und keine Ausgabe, was gelöscht wird. Besser: erst nur **ausgeben**, was gelöscht würde, und mit einer Person abstimmen. Löschen gehört in die Verantwortung der Betriebs-IT.
- Die Pfade sind relativ zum Arbeitsverzeichnis. Läuft das Skript in einem anderen Ordner, trifft es andere Dateien.
</details>

<details><summary>Lösung 3.5 Eigene Auswertung: Dateien nach Endung zählen</summary>

```python
from pathlib import Path

ordner = Path("zelle")
anzahl_je_endung = {}
for p in sorted(ordner.rglob("*")):
    if p.is_file():
        endung = p.suffix
        if endung not in anzahl_je_endung:
            anzahl_je_endung[endung] = 0
        anzahl_je_endung[endung] = anzahl_je_endung[endung] + 1
print(anzahl_je_endung)
```
</details>

## Rückblick
- `from pathlib import Path`: `Path("zelle") / "maschinen"` baut Pfade betriebssystemunabhängig. `.name`, `.stem`, `.suffix` und `.parent` zerlegen einen Pfad, `.exists()`, `.is_file()` und `.is_dir()` prüfen ihn.
- `iterdir()` listet einen Ordner, `glob("maschinen/*/logs/*.log")` sucht mit Muster (ein Stern je Ordnerebene), `rglob("*.nc")` geht durch alle Unterordner. Die Reihenfolge ist nicht festgelegt: **`sorted(...)`** macht sie fest.
- `stat().st_size` liefert die Größe in Bytes, durch 1024 geteilt sind das KiB. Kurze Dateien liest `read_text(encoding="utf-8")`, lange Logs liest du weiter mit `with open` und einer Schleife.
- `os.path.join`, `os.listdir`, `os.getcwd` und `os.walk` musst du **lesen** können (ältere Schreibweise, häufig in KI-Code). Selbst schreibst du es mit `pathlib`.
- `time.perf_counter()` ist eine Stoppuhr: Differenz mal 1000 ergibt Millisekunden. Messwerte schwanken, weil das Betriebssystem andere Prozesse dazwischenschiebt. Mehrmals messen, Bestwert oder Median nehmen.
- Task-Manager lesen: ein Kern bei 100 % und gesamt 13 % heißt, das Programm nutzt nur einen Kern. Fast voller Arbeitsspeicher heißt, die Auslagerung droht. Daten im Netz heißen, jeder Zugriff wartet auf das Netzwerk.
- Code der KI, der Dateien löscht oder überschreibt, wird nie ungeprüft ausgeführt.